# `geoai-datacubes` — A grand tour

<a href="https://colab.research.google.com/github/buckai-observatory/geoai-datacubes/blob/main/notebooks/00_geoai_datacubes_tour.ipynb" target="_blank" rel="noopener noreferrer"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


**Welcome!** This notebook is intended to give a full overview of the `geoai-datacubes`
pipeline for users with no prior geospatial-ML background. `geoai-datacubes` can download satellite data for you from a wide range of different satellite (including Sentinel-2 (L2A and L1C), Sentinel-1
 RTC, Landsat, Copernicus DEM, ESA WorldCover, PlanetScope and more), fuse multimodal satellite data, and preprocess into AI-ready data-cubes.

Run it top to bottom on a
fresh checkout and you will have built — from scratch, with no credentials —
a multi-mission, multi-resolution, AI-ready satellite data cube. This notebook uses a region around the Ohio State University campus in
Columbus, OH as an example.

#### What this notebook covers
1. Setup and environment
2. The four ways to define an Area of Interest (AOI)
3. Fetching one mission at a time — examples: Sentinel-2 (L2A and L1C), Sentinel-1
 RTC, Landsat, Copernicus DEM, ESA WorldCover
4. The commercial path — PlanetScope (4-band and 8-band SuperDove)
5. Cloud masking up close (SCL, BQA, UDM2)
6. NaN handling — `drop`, `interpolate`, `mask`
7. Tiling — with and without overlap, visualised
8. Train / val / test splits — `random`, `block`, `stripes`, `regions`
9. Multi-mission fusion onto a common UTM grid
10. Reading metadata back out from tiles (GeoTIFF tags + sidecars)
11. Augmentation
12. Exporting for training (Zarr / LMDB)
13. Doing the same from plain Python (`python main.py`)
14. Running on an HPC cluster (SLURM)
15. Where to go next

#### How to read this notebook
Each section opens with a short markdown explanation of *why* a novice
would care, then the code, then commentary on the numbers and the figure.
We deliberately print shapes, CRS, pixel sizes, NaN fractions, and split
counts everywhere — sanity checks make the difference between a model
that works and a model that *looks like* it works.

#### A note on running things from plain Python
Every cell here has an exact equivalent at the bottom of
`geoai_datacubes/main.py`. The notebook is great for exploration;
for production runs (HPC, Airflow, cron) you want plain Python. See
section 13 for the equivalence and `slurm_examples/` for SLURM templates.
The cluster-specific bits (partition names, modules, JupyterHub URLs)
live in the **BuckAI HPC Handbook** (https://buckai-observatory.org/buckai-hpc-handbook/).

## 1. Setup and environment

This notebook lives at `notebooks/00_geoai_datacubes_tour.ipynb` in the
repo. We assume you already created the conda env and installed the
requirements as documented in the [top-level README](../README.md):

```bash
conda create -n geoai python=3.11 -y
conda activate geoai
pip install -r requirements.txt
```

#### No credentials needed
The default provider (`"auto"`) routes each mission to a free, public
endpoint — Earth Search (Element 84) for Sentinel-2 and DEM, Planetary
Computer (Microsoft) for Sentinel-1 RTC, Landsat, and ESA WorldCover.
None of these public datasets require an API key or account.

#### Optional credentials
If you want to try the commercial PlanetScope path (Section 4) or the
advanced Sentinel Hub provider, copy `.env.example` to `.env` at the
repo root and fill in `PL_API_KEY` or `SH_CLIENT_ID` / `SH_CLIENT_SECRET`.
Sections that need credentials guard against their absence and simply
skip, so the notebook still runs end-to-end without them.

The next cell adds `geoai_datacubes/` to `sys.path` so we can
`import` the pipeline directly. We also create a `notebooks/_outputs/`
scratch folder where every download, tile, and figure from this notebook
will land — this keeps the rest of the repo clean and makes it easy to
`rm -rf` the outputs if you want to start over.

> **Running on Google Colab?** This notebook is self-bootstrapping — open the file in Colab via
> [`colab.research.google.com/github/buckai-observatory/geoai-datacubes/blob/main/notebooks/00_geoai_datacubes_tour.ipynb`](https://colab.research.google.com/github/buckai-observatory/geoai-datacubes/blob/main/notebooks/00_geoai_datacubes_tour.ipynb)
> and just run the cells. The next cell detects Colab, clones the repo into `/content/geoai-datacubes`, installs the handful of packages not in Colab's default image, and chdir's so the rest of the notebook works unchanged.
> On a local Jupyter run that same cell is a quick no-op.


In [ ]:
# --- Colab / local bootstrap ---
# On Colab, this cell clones the repo and installs the few packages that
# are not in Colab's default runtime. On a local Jupyter run it is a no-op
# that just confirms the repo layout. Re-running it is safe.

import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected — bootstrapping repo + dependencies")
    REPO_DIR = Path("/content/geoai-datacubes")
    if not REPO_DIR.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/buckai-observatory/geoai-datacubes.git",
            str(REPO_DIR),
        ])
    else:
        print(f"repo already cloned at {REPO_DIR}")

    # Colab pre-installs rasterio / shapely / scikit-image / scipy / tqdm /
    # pandas / pyproj / requests / matplotlib. These extras may be missing:
    missing = []
    for pkg, importname in [("python-dotenv", "dotenv"),
                            ("pystac",        "pystac"),
                            ("contextily",    "contextily")]:
        try:
            __import__(importname)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"pip install -q {' '.join(missing)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

    # Land in the notebooks/ folder so the next cell's parent-search picks
    # up the cloned repo cleanly.
    os.chdir(REPO_DIR / "notebooks")
    print(f"cwd = {os.getcwd()}")
else:
    print("Local environment — using existing checkout")


The next cell imports some necessary python libraries, tells the notebook where the geoai-datacubes/ code lives, and specifies a structure of folders where to download imagery and stage our AI-ready datacubes. 

In [ ]:
# --- imports + path setup ---
import glob, os, sys, json, glob, shutil, time, math, warnings
from pathlib import Path

import numpy as np
import rasterio
from rasterio.warp import transform_bounds
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap, Normalize

# locate the repo root from this notebook's CWD (notebooks/)
NB_DIR  = Path.cwd()
if NB_DIR.name != "notebooks":
    # safety net: try to find notebooks/ in parents
    for p in (NB_DIR, *NB_DIR.parents):
        if (p / "notebooks").is_dir() and (p / "geoai_datacubes").is_dir():
            NB_DIR = p / "notebooks"
            break
REPO_ROOT = NB_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

# scratch output folder for this notebook
OUT  = NB_DIR / "_outputs"
DATA = OUT / "data"
FUSE = OUT / "fused"
TILES = OUT / "tiles"
for d in (OUT, DATA, FUSE, TILES):
    d.mkdir(parents=True, exist_ok=True)

# nicer matplotlib defaults
plt.rcParams["figure.dpi"]   = 110
plt.rcParams["savefig.dpi"]  = 110
plt.rcParams["image.cmap"]   = "viridis"
plt.rcParams["axes.grid"]    = False
warnings.filterwarnings("ignore", category=UserWarning, module="rasterio")

print("Repo root :", REPO_ROOT)
print("Package   :", REPO_ROOT / "geoai_datacubes")
print("Outputs   :", OUT)

Next, we import the specific geoai-datacubes routines that will be used in this notebook.

In [ ]:
# --- pipeline imports (these must work end-to-end on a fresh checkout) ---
from geoai_datacubes.fetch import resolve_aoi
from geoai_datacubes.fetch import get_profile, MISSION_PROFILES
from geoai_datacubes.fetch import fetch_sentinel_data
from geoai_datacubes.preprocessing import normalize_band, compute_ndvi, cloud_mask
from geoai_datacubes.preprocessing import tile_geotiff
from geoai_datacubes.preprocessing import fuse_response_tiffs

# Simple print
# print("Available missions:", sorted(MISSION_PROFILES.keys()))

# Pretty-print the mission registry as an aligned table.
rows = []
for name in sorted(MISSION_PROFILES):
  p = MISSION_PROFILES[name]
  bands = ", ".join(p.get("default_bands", []))
  providers = ", ".join(sorted(p.get("providers", {}).keys()))
  static = "yes" if p.get("static") else ""
  rows.append((name, bands, static, providers))

print(f"{'Mission':<18}  {'Default bands':<30}  {'Static':<6}  Providers")
print(f"{'-'*18}  {'-'*30}  {'-'*6}  {'-'*40}")
for name, bands, static, providers in rows:
  print(f"{name:<18}  {bands[:30]:<30}  {static:<6}  {providers}")
print(f"\n({len(rows)} missions registered)")

Next, import some helper functions designed to keep notebooks like these cleaner. 

In [ ]:
from geoai_datacubes.viz import (
    # scenes
    open_response, print_response_summary, read_userdata,
    find_response, find_band,
    stretch01, show_bands, show_rgb,
    plot_cloud_pair, decode_scl, decode_bqa,
    # tiles
    count_tiles, open_tile, best_demo_tile, tile_grid_overlay,
    # splits
    SPLIT_COLOURS, read_tiles_csv, plot_split_layout, city_panel,
)


## 2. Defining your Area of Interest (AOI)

The pipeline accepts **four** AOI formats, all resolved by
`aoi.py::resolve_aoi(spec)` to the same WGS84 bbox
`[lon_min, lat_min, lon_max, lat_max]`. Pick whichever is most natural
for your workflow:

| Format | When to use |
|---|---|
| `{"bbox": [lon_min, lat_min, lon_max, lat_max]}` | You already know the corners. |
| `{"shapefile": "path/to/aoi.shp"}` | You have a vector polygon (county, watershed, field boundary). Requires `geopandas`. Uses the polygon's rectangular **bounding box** (because typical DL workflows like convolutional neural networks (CNN) require square/rectangular input clips). |
| `{"center": (lat, lon), "side_miles": N}` | You know roughly where, want a square AOI of size *N* miles. |
| `{"tile_around": (lat, lon)}` | You want the full Sentinel-2 MGRS tile (~100×100 km) containing the point — fastest first look. |

For the rest of this notebook we use a **10-mile square AOI** centred on the OSU campus in
Columbus, OH. The 10-mile box yields plenty of smaller tiles at 64, 128, or 256 pixels, so every train / val / test bucket stays comfortably populated regardless of split method. Each fetch shouldn't take more than a minute depending on how many bands you ask for. Feel free to swap the AOI for a shapefile of a county / watershed / field boundary -- the pipeline accepts any of the four formats above.

In [ ]:
# ============================================================
# USER INPUT
# ============================================================
# This block is the only thing you typically edit at the top of the
# notebook. The defaults below give a 10-mile square over OSU campus,
# Columbus, OH -- large enough that the train/val/test splits later
# end up with a healthy number of tiles in EVERY bucket regardless of
# which split method you choose.

AOI = {"center": (40.0067, -83.0305), "side_miles": 10}   # OSU campus, 10 mi square
ROI = resolve_aoi(AOI)

# ---- Cloud toggle -------------------------------------------------
# WITH_CLOUDS=True picks scenes that contain real cloud cover, so the
# cloud-masking and NaN-handling sections below have something to bite
# on. Cloud-free imagery is the easy part in EO -- the interesting
# part is what your pipeline does when clouds are present. Set this
# to False if you would rather see a pristine cube end-to-end.
WITH_CLOUDS = True

if WITH_CLOUDS:
    # Winter window for Columbus. The MIN_CLOUD floor is the key knob --
    # without it the search just returns the cleanest scene in the
    # window (often <5% cloud), which leaves nothing for the cloud-
    # masking section to demonstrate. With MIN_CLOUD=0.10 we ask the
    # search to skip scenes below 10% cloud cover and prefer the
    # lowest-cloud scene above that floor -- typically 10-20%.
    TIME_RANGE_OPTICAL = ("2024-02-01", "2024-03-31")
    TIME_RANGE_LANDSAT = ("2024-02-01", "2024-04-30")   # 16-day revisit -> wider
    MIN_CLOUD          = 0.10
    MAX_CLOUD          = 0.40
    cloud_label        = "WITH_CLOUDS=True  (winter window, cloud 10-40%)"
else:
    TIME_RANGE_OPTICAL = ("2024-06-15", "2024-06-25")
    TIME_RANGE_LANDSAT = ("2024-05-01", "2024-09-30")
    MIN_CLOUD          = 0.0
    MAX_CLOUD          = 0.10
    cloud_label        = "WITH_CLOUDS=False (summer window, cloud <10%)"

TIME_RANGE_RADAR = ("2024-06-01", "2024-06-30")   # radar unaffected by clouds

# ---- Band selection per mission -----------------------------------
# Pick whichever bands you want from each mission. Mission profiles
# in missions.py document the full list per sensor; the comments
# below show what is available out of the box. For most optical
# work you want at minimum the visible RGB (B02/B03/B04) plus
# the NIR for vegetation work, plus the QA / cloud-class band.
#
# Sentinel-2 L2A:  B01, B02 (blue), B03 (green), B04 (red), B05, B06,
#                  B07, B08 (NIR), B8A, B09, B11 (SWIR1), B12 (SWIR2),
#                  SCL (scene classification), AOT, WVP
# Sentinel-2 L1C:  B01..B12, B8A, visual              (no SCL in L1C)
# Sentinel-1 RTC:  VV, VH                              (HH, HV in EW mode)
# Landsat C2 L2:   B01 (coastal), B02 (blue), B03 (green), B04 (red),
#                  B05 (NIR), B06 (SWIR1), B07 (SWIR2), B10 (thermal),
#                  BQA (cloud bit-mask)
# Copernicus-DEM:  DEM
# ESA-WorldCover:  LULC
BANDS_S2  = ["B02", "B03", "B04", "B08", "SCL"]   # blue/green/red/NIR + cloud-class
BANDS_L1C = ["B02", "B03", "B04", "B08"]           # blue/green/red/NIR
BANDS_S1  = ["VV", "VH"]                            # both polarisations
BANDS_LSC = ["B02", "B03", "B04", "B05", "BQA"]    # blue/green/red/NIR + cloud QA
BANDS_DEM = ["DEM"]
BANDS_LULC= ["LULC"]

# NAIP -- USDA aerial imagery at sub-metre resolution. RGB+NIR. The full
# 10 mi AOI at native 1 m would be roughly 16000x16000 pixels per band
# (~4 GB of imagery) which is uncomfortable for a Colab notebook, so we
# zoom into a smaller sub-AOI for the NAIP demonstration below.
BANDS_NAIP = ["R", "G", "B", "NIR"]
TIME_RANGE_NAIP = ("2022-01-01", "2024-12-31")   # NAIP archive window
AOI_NAIP_ZOOM = {"center": (40.0067, -83.0305), "side_miles": 0.5}

# ---- Output resolutions --------------------------------------------
# Stay at the mission's native ground sampling whenever possible --
# over-resampling either way costs information.
RES_S2  = 10
RES_S1  = 10
RES_LSC = 30
RES_DEM = 30
RES_LULC= 10
RES_NAIP = 1.0    # NAIP native ground sampling (US states acquired post-2018 fly at 0.6 m)

print(f"Working AOI : {ROI}")
print(f"size       : ~{(ROI[3]-ROI[1])*111:.2f} km tall x"
      f"{(ROI[2]-ROI[0])*111*math.cos(math.radians(ROI[1])):.2f} km wide")
print(f"Cloud mode  : {cloud_label}")
cloud_range = (f"cloud={int(MIN_CLOUD*100)}-{int(MAX_CLOUD*100)}%"
               if MIN_CLOUD > 0 else f"max_cloud={int(MAX_CLOUD*100)}%")
print(f"optical    : {TIME_RANGE_OPTICAL[0]} -> {TIME_RANGE_OPTICAL[1]}  ({cloud_range})")
print(f"Landsat    : {TIME_RANGE_LANDSAT[0]} -> {TIME_RANGE_LANDSAT[1]}")
print(f"radar      : {TIME_RANGE_RADAR[0]} -> {TIME_RANGE_RADAR[1]}")
print(f"Bands       : S2={BANDS_S2}  S2-L1C={BANDS_L1C}")
print(f"S1={BANDS_S1}  Landsat={BANDS_LSC}")
print(f"DEM={BANDS_DEM}  LULC={BANDS_LULC}")


In [ ]:
# --- demonstrate all four AOI formats ---
specs = {
    "bbox            (example: small downtown Columbus box)" :
        {"bbox": [-83.020, 39.995, -83.000, 40.015]},

    "center+side_miles (example: 5 mi square around OSU)" :
        {"center": (40.0067, -83.0305), "side_miles": 5},

    "tile_around     (example: native S2 MGRS tile @ OSU)" :
        {"tile_around": (40.0067, -83.0305)},

    # shapefile demo: write a 1-feature GeoJSON to disk and resolve it.
    # If geopandas is missing this AOI is skipped (caught below).
}

# --- write a 1-feature GeoJSON so we can also demonstrate the shapefile format ---
gj_path = OUT / "demo_aoi.geojson"
demo_box = [-83.080, 39.990, -83.010, 40.025]
with open(gj_path, "w") as f:
    json.dump({
        "type": "FeatureCollection",
        "features": [{
            "type": "Feature",
            "properties": {"name": "demo_box"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [demo_box[0], demo_box[1]],
                    [demo_box[2], demo_box[1]],
                    [demo_box[2], demo_box[3]],
                    [demo_box[0], demo_box[3]],
                    [demo_box[0], demo_box[1]],
                ]],
            },
        }],
    }, f)
specs["shapefile       (example: 1-feature GeoJSON, bbox used)"] = {"shapefile": str(gj_path)}

# --- resolve each spec to a WGS84 bbox ---
resolved = {}
for label, spec in specs.items():
    try:
        resolved[label] = resolve_aoi(spec)
    except Exception as e:
        resolved[label] = f"(skipped: {type(e).__name__}: {e})"

print(f"{'Format':52s}  bbox  [lon_min, lat_min, lon_max, lat_max]")
print("-" * 100)
for label, bbox in resolved.items():
    if isinstance(bbox, list):
        side_km = 111 * (bbox[3] - bbox[1])
        side_mi = 69 * (bbox[3] - bbox[1])
        print(f"{label:52s}  [{bbox[0]:8.4f}, {bbox[1]:7.4f},"
              f"{bbox[2]:8.4f}, {bbox[3]:7.4f}]  ({side_mi:.1f} mi / {side_km:.1f} km tall)")
    else:
        print(f"{label:52s}  {bbox}")

In [ ]:
# --- plot the four AOI formats + the working AOI over an OpenStreetMap basemap ---
# Falls back to a plain lon/lat grid if contextily is not available.

try:
    import contextily as cx
    HAVE_CTX = True
except ImportError:
    HAVE_CTX = False

fig, ax = plt.subplots(figsize=(9, 9))

# Frame: tight enough that the working AOI is comfortably visible, loose
# enough that the larger format examples (S2 tile, OSU 5 mi square) fit.
pad_lat = max(0.15, (ROI[3] - ROI[1]) * 1.5)
pad_lon = max(0.18, (ROI[2] - ROI[0]) * 1.5)
cx0 = (ROI[0] + ROI[2]) / 2
cy0 = (ROI[1] + ROI[3]) / 2
frame = (cx0 - pad_lon, cx0 + pad_lon, cy0 - pad_lat, cy0 + pad_lat)

# OSU campus marker
ax.plot(-83.0305, 40.0067, "o", color="#BA0C2F", ms=8, zorder=5,
        label="OSU campus (40.0067°N, 83.0305°W)")

# The four example AOIs in muted colours (instructional -- not used downstream)
example_colors = ["#1f77b4", "#2ca02c", "#9467bd", "#ff7f0e"]
for (label, bbox), c in zip(resolved.items(), example_colors):
    if not isinstance(bbox, list):
        continue
    rect = Rectangle(
        (bbox[0], bbox[1]),
        bbox[2] - bbox[0],
        bbox[3] - bbox[1],
        fill=False, edgecolor=c, lw=1.5, ls="--", zorder=3,
        label=label.split()[0] + "  (example)",
    )
    ax.add_patch(rect)

# THE working AOI -- the one the rest of the notebook actually uses --
# drawn thicker and in red so it stands out from the dashed examples.
working_rect = Rectangle(
    (ROI[0], ROI[1]),
    ROI[2] - ROI[0],
    ROI[3] - ROI[1],
    fill=False, edgecolor="#BA0C2F", lw=3.0, zorder=6,
    label="working AOI (used downstream)",
)
ax.add_patch(working_rect)

ax.set_xlim(frame[0], frame[1])
ax.set_ylim(frame[2], frame[3])
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_aspect("equal")
ax.set_title("Four AOI formats (dashed examples)\nplus the working AOI (red, used by sections 3+)",
             fontsize=11)

if HAVE_CTX:
    try:
        cx.add_basemap(ax, crs="EPSG:4326",
                       source=cx.providers.OpenStreetMap.Mapnik,
                       attribution_size=6)
    except Exception as e:
        print(f"(basemap unavailable: {e}; falling back to plain grid)")
        ax.grid(True, ls=":", alpha=0.4)
else:
    print("(contextily not installed; using a plain lon/lat grid)")
    ax.grid(True, ls=":", alpha=0.4)

ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.show()


The figure makes the trade-offs concrete:

- **`tile_around`** returns the entire Sentinel-2 MGRS tile (~100 × 100 km).
 Largest area, biggest download, but you only need one point to specify it.
- **`center` + `side_miles`** gives you a clean square of any size — a great
 default when you know the rough location.
- **`shapefile`** lets you reuse polygons you already have (counties,
 ecoregions, study sites). Note that the pipeline uses the polygon's
 **bounding box**, not the polygon itself; if you need to mask outside the
 polygon, do that downstream after tiling.
- **`bbox`** is the most explicit format and the one we use below.

The **red rectangle** above is the *working AOI* — the one cell 6 defined and the one every section below pulls data over. Resize / relocate it there to point the rest of this notebook anywhere on Earth.

## 3. One mission at a time

Every fetch produces three things in `_outputs/data/<Mission>_<date>_<scene_id>/`:

1. `<Mission>_full_size.tiff` — a multi-band GeoTIFF on the AOI's local UTM
 grid at the requested resolution. **One channel per band you listed in
 `BANDS_<mission>`** in cell 6 — nothing more, nothing less. (If you
 pass `bands=None` from plain Python you get the mission's default set
 plus its helper bands like SCL or BQA; the notebook above is explicit.)
2. `userdata.json` — a small sidecar with the scene's metadata
 (satellite, date, cloud cover, scene id, provider, collection).
3. Optional NDVI PNGs when you run `main.py` — we'll skip those here and
 compute NDVI ourselves so you can see the math.

Below we walk through several of the **free** missions that are widely used. Each subsection follows the
same recipe: fetch → open → print band names + CRS + pixel size + NaN
fraction → plot. If a fetch fails (e.g. cloudy week, network glitch) we
catch the exception, print it, and continue — the notebook keeps going.

### 3a. Sentinel-2 L2A — surface reflectance (the workhorse)

Sentinel-2 L2A is bottom-of-atmosphere surface reflectance, 10 m native
ground sampling for the visible/NIR bands. By default the notebook fetches
the bands listed in `BANDS_S2` in cell 6 — currently the blue/green/red/NIR
set plus the SCL scene-classification layer, which is enough for true-colour
RGB previews, NDVI, and per-pixel cloud masking. Swap the band list any
time — see `missions.py` for the full alphabet that L2A exposes.

In [ ]:
%%time
# --- fetch Sentinel-2 L2A ---
print(">>> Sentinel-2 (L2A) <<<")
t0 = time.time()
_data, _bands = fetch_sentinel_data(
    "Sentinel-2", BANDS_S2, TIME_RANGE_OPTICAL, ROI,
    resolution=RES_S2, save_folder=str(DATA),
    max_cloud_coverage=MAX_CLOUD, min_cloud_coverage=MIN_CLOUD, provider="auto",
)
print(f"fetched in {time.time()-t0:.1f}s\n")
S2_DIR = sorted(DATA.glob("Sentinel-2_*"), key=os.path.getmtime)[-1]
print("scene folder:", S2_DIR.name, "\n")
arr_s2, descs_s2, prof_s2, bbox_s2, xform_s2, crs_s2, tiff_s2 = open_response(S2_DIR)
print_response_summary(arr_s2, descs_s2, prof_s2, bbox_s2, xform_s2, crs_s2, tiff_s2)
print("\nuserdata.json:")
print(json.dumps(read_userdata(S2_DIR), indent=2))

In [ ]:
%%time
# --- visualise: true-colour RGB, then every individual band ---
show_rgb(arr_s2, descs_s2, f"Sentinel-2 L2A (true colour) — {S2_DIR.name}")

cmap_s2 = {"B02": "gray", "B03": "gray", "B04": "gray", "B08": "gray",
           "SCL": "tab20", "AOT": "viridis", "WVP": "viridis"}
show_bands(arr_s2, descs_s2,
           f"Sentinel-2 L2A — {S2_DIR.name}",
           cmap_per_band=cmap_s2)


The **SCL** (Scene Classification Layer) band uses integer class codes
3 = cloud shadow, 8/9 = cloud, 10 = cirrus, etc. — that's what the tiler
will use in Section 5 to mask cloudy pixels. **AOT** (aerosol optical
thickness) and **WVP** (water vapour) are atmospheric helpers; you usually
don't train on them directly but they're handy diagnostics.

### 3b. Sentinel-2 L1C — top-of-atmosphere

L1C is *top-of-atmosphere* reflectance — the same image as L2A but without
the atmospheric correction (and without SCL, so no per-pixel cloud mask).
Useful when you want to do your own atmospheric correction, train on
foundation models that were pre-trained on L1C, or push back to the early
years of the archive when L2A was patchy.

In [ ]:
%%time
print(">>> Sentinel-2 L1C <<<")
t0 = time.time()
try:
    _data, _bands = fetch_sentinel_data(
        "Sentinel-2-L1C", BANDS_L1C, TIME_RANGE_OPTICAL, ROI,
        resolution=RES_S2, save_folder=str(DATA),
        max_cloud_coverage=MAX_CLOUD, min_cloud_coverage=MIN_CLOUD, provider="auto",
    )
    print(f"fetched in {time.time()-t0:.1f}s\n")
    L1C_DIR = sorted(DATA.glob("Sentinel-2-L1C_*"), key=os.path.getmtime)[-1]
    print("scene folder:", L1C_DIR.name, "\n")
    arr_l1c, descs_l1c, prof_l1c, bb_l1c, xf_l1c, crs_l1c, tf_l1c = open_response(L1C_DIR)
    print_response_summary(arr_l1c, descs_l1c, prof_l1c, bb_l1c, xf_l1c, crs_l1c, tf_l1c)
    show_rgb(arr_l1c, descs_l1c, f"Sentinel-2 L1C (true colour) — {L1C_DIR.name}")
    show_bands(arr_l1c, descs_l1c, f"Sentinel-2 L1C — {L1C_DIR.name}",
               cmap_per_band={b: "gray" for b in descs_l1c})
except Exception as e:
    print(f"L1C fetch skipped: {type(e).__name__}: {e}")
    L1C_DIR = None

### 3c. Sentinel-1 RTC — SAR backscatter (sees through clouds)

Sentinel-1 is **synthetic-aperture radar (SAR)** — active microwave imaging
that works at night and through clouds. The pipeline pulls the
**radiometric-terrain-corrected (RTC)** product from Planetary Computer
(Earth Search only has raw GRD, which lacks a usable CRS). The two bands
are dual-polarisation backscatter:

- **VV** — vertical transmit, vertical receive. Bright on rough surfaces.
- **VH** — vertical transmit, horizontal receive. Sensitive to vegetation
 volume scattering.

Values are linear (not dB). For visualisation we stretch each band to
percentile [2, 98] so the urban/water/vegetation contrast pops out.

In [ ]:
%%time
print(">>> Sentinel-1 RTC <<<")
t0 = time.time()
try:
    _data, _bands = fetch_sentinel_data(
        "Sentinel-1", BANDS_S1, TIME_RANGE_RADAR, ROI,
        resolution=RES_S1, save_folder=str(DATA),
        max_cloud_coverage=MAX_CLOUD,    # ignored for SAR -- radar sees through clouds
        provider="auto",
    )
    print(f"fetched in {time.time()-t0:.1f}s")
    S1_DIR = sorted(DATA.glob("Sentinel-1_*"), key=os.path.getmtime)[-1]
    arr_s1, descs_s1, prof_s1, bb_s1, xf_s1, crs_s1, tf_s1 = open_response(S1_DIR)
    print_response_summary(arr_s1, descs_s1, prof_s1, bb_s1, xf_s1, crs_s1, tf_s1)
    show_bands(arr_s1, descs_s1, f"Sentinel-1 RTC — {S1_DIR.name}",
               cmap_per_band={b: "gray" for b in descs_s1})
except Exception as e:
    print(f"Sentinel-1 fetch skipped: {type(e).__name__}: {e}")
    S1_DIR = None

### 3d. Landsat 8-9 Collection 2 Level-2

Landsat-8 and -9 fly together with a combined 8-day revisit. The pipeline
fetches the bands listed in `BANDS_LSC` (cell 6) at 30 m. The default set
is blue/green/red/NIR plus `BQA` — which is enough for a true-colour
preview, NDVI (B04/B05), and per-pixel cloud / shadow masking. The BQA (`qa_pixel`)
band is a **bit-packed** quality mask — bit 1 = dilated cloud, bit 3 =
cloud, bit 4 = cloud shadow. The tiler decodes this for cloud masking in
Section 5.

In [ ]:
print(">>> Landsat 8/9 C2 L2 <<<")
t0 = time.time()
try:
    _data, _bands = fetch_sentinel_data(
        "Landsat", BANDS_LSC, TIME_RANGE_LANDSAT, ROI,
        resolution=RES_LSC, save_folder=str(DATA),
        max_cloud_coverage=MAX_CLOUD, min_cloud_coverage=MIN_CLOUD, provider="auto",  # honors the WITH_CLOUDS toggle
    )
    print(f"fetched in {time.time()-t0:.1f}s")
    LSC_DIR = sorted(DATA.glob("Landsat_*"), key=os.path.getmtime)[-1]
    arr_lsc, descs_lsc, prof_lsc, bb_lsc, xf_lsc, crs_lsc, tf_lsc = open_response(LSC_DIR)
    print_response_summary(arr_lsc, descs_lsc, prof_lsc, bb_lsc, xf_lsc, crs_lsc, tf_lsc)
    show_rgb(arr_lsc, descs_lsc, f"Landsat C2 L2 (true colour) — {LSC_DIR.name}",
             r_band="B04", g_band="B03", b_band="B02")
    show_bands(arr_lsc, descs_lsc, f"Landsat C2 L2 — {LSC_DIR.name}",
               cmap_per_band={"B02": "gray", "B03": "gray", "B04": "gray",
                              "B05": "gray", "BQA": "tab20c"})
except Exception as e:
    print(f"Landsat fetch skipped: {type(e).__name__}: {e}")
    LSC_DIR = None

### 3e. Copernicus DEM — 30 m global elevation (static)

DEMs are *static* layers — there's no time component. The pipeline ignores
`TIME_RANGE` for static missions, deduplicates by spatial tile, and
mosaics whatever tiles intersect the AOI onto the requested grid. The
single band is just **DEM** (elevation in metres above ellipsoid).

In [ ]:
print(">>> Copernicus DEM <<<")
t0 = time.time()
_data, _bands = fetch_sentinel_data(
    "Copernicus-DEM", BANDS_DEM, TIME_RANGE_OPTICAL, ROI,
    resolution=RES_DEM, save_folder=str(DATA),
    max_cloud_coverage=MAX_CLOUD, provider="auto",
)
print(f"fetched in {time.time()-t0:.1f}s")
DEM_DIR = sorted(DATA.glob("Copernicus-DEM_*"), key=os.path.getmtime)[-1]
arr_dem, descs_dem, prof_dem, bb_dem, xf_dem, crs_dem, tf_dem = open_response(DEM_DIR)
print_response_summary(arr_dem, descs_dem, prof_dem, bb_dem, xf_dem, crs_dem, tf_dem)

# Richer DEM render: combine a sun-shaded relief (hillshade) with the
# terrain colormap *and* drop iso-elevation contours over the top. A flat
# colormap of elevation is technically correct but visually flat; this
# version reads like a real topo map.
from matplotlib.colors import LightSource

z = arr_dem[0].astype(np.float32)
# Sun position: from the northwest, low in the sky -- classic cartographic
# convention so north faces appear bright and shadows fall southeast.
ls = LightSource(azdeg=315, altdeg=45)
shaded_rgb = ls.shade(z, cmap=plt.cm.terrain, vert_exag=8, blend_mode="soft")

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
ax.imshow(shaded_rgb)
# Pick a contour interval that yields ~10-20 labelled lines across the
# elevation range -- on the 16 km Columbus AOI this lands around every
# 5 m of elevation (the Scioto valley is gentle).
zmin, zmax = float(np.nanmin(z)), float(np.nanmax(z))
n_levels = 12
step = max(1.0, round((zmax - zmin) / n_levels))
levels = np.arange(np.floor(zmin / step) * step,
                   zmax + step, step)
cs = ax.contour(z, levels=levels, colors="black",
                linewidths=0.4, alpha=0.7)
ax.clabel(cs, inline=True, fontsize=6, fmt="%d m")

# Colourbar showing the elevation scale that drives the colormap.
sm = plt.cm.ScalarMappable(cmap=plt.cm.terrain,
                           norm=plt.Normalize(vmin=zmin, vmax=zmax))
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Elevation (m)", shrink=0.85)

ax.set_title(f"Copernicus DEM — {DEM_DIR.name}\n"
             f"hillshade (sun NW, alt 45°) + {step:g} m contours  |  CRS: {crs_dem}",
             fontsize=10)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

Notice in the metadata block that `static = True` and `mosaic_tiles ≥ 1`:
the pipeline pulled however many 1° DEM tiles touched our AOI and welded
them together onto the output grid. Same applies to WorldCover below.

### 3f. ESA WorldCover — 10 m global land cover (static)

WorldCover is a global, **categorical** 10 m land-cover map (2020 + 2021
versions). The pipeline picks the latest version per tile and uses
**nearest-neighbour resampling** so class IDs are preserved exactly
(10 = tree cover, 20 = shrubland, 30 = grassland, 40 = cropland,
50 = built-up, 60 = bare/sparse vegetation, 70 = snow/ice, 80 = permanent
water, 90 = herbaceous wetland, 95 = mangroves, 100 = moss/lichen).

In [ ]:
print(">>> ESA WorldCover <<<")
t0 = time.time()
_data, _bands = fetch_sentinel_data(
    "ESA-WorldCover", BANDS_LULC, TIME_RANGE_OPTICAL, ROI,
    resolution=RES_LULC, save_folder=str(DATA),
    max_cloud_coverage=MAX_CLOUD, provider="auto",
)
print(f"fetched in {time.time()-t0:.1f}s")
LULC_DIR = sorted(DATA.glob("ESA-WorldCover_*"), key=os.path.getmtime)[-1]
arr_lulc, descs_lulc, prof_lulc, bb_lulc, xf_lulc, crs_lulc, tf_lulc = open_response(LULC_DIR)
print_response_summary(arr_lulc, descs_lulc, prof_lulc, bb_lulc, xf_lulc, crs_lulc, tf_lulc)

# --- a categorical palette + legend ---
LULC_CLASSES = {
    10: ("Tree cover",       "#006400"),
    20: ("Shrubland",        "#FFBB22"),
    30: ("Grassland",        "#FFFF4C"),
    40: ("Cropland",         "#F096FF"),
    50: ("Built-up",         "#FA0000"),
    60: ("Bare / sparse veg","#B4B4B4"),
    70: ("Snow / ice",       "#F0F0F0"),
    80: ("Water",            "#0064C8"),
    90: ("Herbaceous wetland","#0096A0"),
    95: ("Mangroves",        "#00CF75"),
    100:("Moss / lichen",    "#FAE6A0"),
}
codes = sorted(LULC_CLASSES.keys())
code_to_idx = {c: i for i, c in enumerate(codes)}
arr_idx = np.full(arr_lulc[0].shape, -1, dtype=int)
for c, i in code_to_idx.items():
    arr_idx[arr_lulc[0] == c] = i

palette = ListedColormap([LULC_CLASSES[c][1] for c in codes])
palette.set_bad("white")  # for -1 / NaN
masked = np.ma.masked_less(arr_idx, 0)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.imshow(masked, cmap=palette, vmin=-0.5, vmax=len(codes) - 0.5, interpolation="nearest")
ax.set_title(f"ESA WorldCover — {LULC_DIR.name}\nCRS: {crs_lulc}")
ax.set_xticks([]); ax.set_yticks([])

# class legend
from matplotlib.patches import Patch
handles = [Patch(facecolor=LULC_CLASSES[c][1], edgecolor="k",
                 label=f"{c}: {LULC_CLASSES[c][0]}") for c in codes]
ax.legend(handles=handles, bbox_to_anchor=(1.02, 1.0), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

# class-wise pixel counts so we can sanity-check the figure
unique, counts = np.unique(arr_lulc[0], return_counts=True)
print("\nPixel counts by class:")
for u, n in zip(unique, counts):
    u = int(u)
    name = LULC_CLASSES.get(u, ("?", ""))[0]
    print(f"{u:>4d}  {name:<20s}  {n:>7d}  ({100*n/counts.sum():5.1f}%)")

### 3g. NAIP -- the resolution leap (sub-metre US aerial imagery)

The first six missions in this section are all *satellite-based*. NAIP
is the odd one out -- it is **airborne imagery** flown by the USDA over
the conterminous US every 2-3 years per state, at **0.6-1 m ground
sampling**. That is an order of magnitude finer than even PlanetScope
and ten times finer than Sentinel-2 -- and it is fully public domain,
no API key needed.

Because NAIP is so much higher resolution, fetching the full 10-mile
Columbus AOI at native 1 m would produce roughly a 16000 x 16000-pixel
image per band (~4 GB total). That is not what you want inside a Colab
notebook just to *see* the imagery, so we zoom into a 0.5 mi sub-AOI
centred on the same OSU spot (`AOI_NAIP_ZOOM` in the USER INPUT cell)
for this preview. For real applications -- especially object detection,
where the 10x resolution advantage really matters -- you would tile
NAIP at full resolution but over a more modest AOI.

Compare the visual richness below with the Sentinel-2 panel above: at
1 m you can read out individual buildings, cars, lawn-mower stripes on
the OSU oval, and the painted lane lines on the streets. At 10 m all
of that has collapsed into one or two pixels of "urban texture".


In [ ]:
print(">>> NAIP <<<")
ROI_NAIP = resolve_aoi(AOI_NAIP_ZOOM)
print(f"   NAIP zoom AOI: {ROI_NAIP}  (~0.8 km square)")

t0 = time.time()
try:
    _data, _bands = fetch_sentinel_data(
        "NAIP", BANDS_NAIP, TIME_RANGE_NAIP, ROI_NAIP,
        resolution=RES_NAIP, save_folder=str(DATA),
        provider="auto",
    )
    print(f"fetched in {time.time()-t0:.1f}s")
    NAIP_DIR = sorted(DATA.glob("NAIP_*"), key=os.path.getmtime)[-1]
    arr_naip, descs_naip, prof_naip, bb_naip, xf_naip, crs_naip, tf_naip = open_response(NAIP_DIR)
    print_response_summary(arr_naip, descs_naip, prof_naip, bb_naip, xf_naip, crs_naip, tf_naip)

    # True-colour preview. NAIP is uint8 0-255 so we let stretch01
    # handle the normalisation just like any other RGB source.
    show_rgb(arr_naip, descs_naip,
             f"NAIP (true colour, ~1 m) -- {NAIP_DIR.name}",
             r_band="R", g_band="G", b_band="B")
    show_bands(arr_naip, descs_naip, f"NAIP -- {NAIP_DIR.name}",
               cmap_per_band={b: "gray" for b in descs_naip})
except Exception as e:
    print(f"NAIP fetch skipped: {type(e).__name__}: {e}")
    NAIP_DIR = None


## 4. The commercial path — PlanetScope

PlanetScope is **commercial** ~3 m optical imagery from Planet Labs. The
free path stops here — to follow along you need an API key in `.env` as
`PL_API_KEY=...`. The pipeline ships two profiles:

- **`PlanetScope-4b`** — the legacy 4-band Dove/SuperDove archive
 (B, G, R, NIR), back to ~2016. Best for long time series.
- **`PlanetScope-8b`** — SuperDove only (PSB.SD, March 2022+) with
 eight bands (Coastal Blue, B, Green I, G, Yellow, R, RedEdge, NIR).
 Spectral coverage close to Sentinel-2's; the right choice for modern
 multispectral modeling.

Both come with a **UDM2** (Usable Data Mask v2) raster: 8 categorical
bands telling you whether each pixel is clear / snow / shadow / haze /
cloud / etc. The pipeline adds `udm2_clear` to the response cube so the
tiler can cloud-mask just like it does for SCL and BQA.

The cell below runs only if `PL_API_KEY` is set — otherwise it prints a
short explanation and moves on. Planet orders are **asynchronous**
(submit -> poll -> download), so expect several minutes if you enable it.

In [ ]:
PL_OK = bool(os.environ.get("PL_API_KEY"))
if not PL_OK:
    print("PL_API_KEY is not set — PlanetScope cells are skipped.")
    print("To try them, get a key from https://www.planet.com/account/#/user-settings")
    print("and put it in a `.env` file at the repo root:  PL_API_KEY=...")
    print("Note: Planet orders are asynchronous and can take several minutes.")
else:
    print("PL_API_KEY detected — running PlanetScope-4b fetch.")
    try:
        t0 = time.time()
        _data, _bands = fetch_sentinel_data(
            "PlanetScope-4b", None, TIME_RANGE_OPTICAL, ROI,
            resolution=3, save_folder=str(DATA),
            max_cloud_coverage=MAX_CLOUD, provider="planet",
        )
        print(f"fetched in {time.time()-t0:.1f}s")
        PS_DIR = sorted(DATA.glob("PlanetScope-4b_*"), key=os.path.getmtime)[-1]
        arr_ps, descs_ps, prof_ps, bb_ps, xf_ps, crs_ps, tf_ps = open_response(PS_DIR)
        print_response_summary(arr_ps, descs_ps, prof_ps, bb_ps, xf_ps, crs_ps, tf_ps)
        show_bands(arr_ps, descs_ps, f"PlanetScope-4b — {PS_DIR.name}",
                   cmap_per_band={"R": "gray", "NIR": "gray",
                                  "udm2_clear": "gray",
                                  "udm2_shadow": "gray",
                                  "udm2_cloud": "gray"})
    except Exception as e:
        print(f"PlanetScope fetch failed: {type(e).__name__}: {e}")

## 5. Cloud masking up close

Two of our missions — Sentinel-2 L2A and Landsat — ship a per-pixel
quality band that flags clouds and shadows. The pipeline knows how to
decode both. Let's visualise them next to a stretched intensity image so
you can see exactly which pixels are about to be masked.

- **SCL** (Sentinel-2 L2A scene classification): integer class codes.
 Classes 3 / 8 / 9 / 10 = shadow / cloud / cloud high / cirrus.
- **BQA / qa_pixel** (Landsat C2 L2): bit-packed. Bits 1 / 3 / 4 =
 dilated cloud / cloud / cloud shadow.
- **UDM2 band 1** (PlanetScope): 1 = clear, 0 = anything else.

In [ ]:
# --- decode the Sentinel-2 SCL cloud / shadow classes and visualise ---
mask_s2 = plot_cloud_pair("Sentinel-2 L2A", arr_s2, descs_s2,
                          intensity_band="B04", qa_band="SCL",
                          decode_qa=decode_scl)


# --- Landsat BQA (only if Landsat fetched OK) ---
if LSC_DIR is not None and "BQA" in descs_lsc:
    mask_lsc = plot_cloud_pair("Landsat", arr_lsc, descs_lsc,
                               intensity_band="B04", qa_band="BQA",
                               decode_qa=decode_bqa)


The middle panel shows the raw quality band — it's most useful for
debugging (you can see *which* class triggered the mask). The right panel
is the binary mask the tiler will apply when you pass `cloud_mask=True`:
those pixels get set to NaN in the data bands (the QA band itself is
preserved). NaN-handling then decides what happens to the tile: drop it,
fill the holes, or keep the tile and append a validity-mask band — see
Section 6.

## 6. NaN handling — `drop`, `interpolate`, `mask`

After cloud masking (or just at the edges of a scene mosaic) you'll have
tiles with some NaN pixels. `tile_geotiff` offers three policies:

- **`drop`** — strict. Any NaN in a tile and the whole tile is skipped.
 Best when you want a perfectly clean training set, can afford to lose
 some area, and don't want to pollute models with imputed values.
- **`interpolate`** — fill NaN pixels from their nearest valid neighbour
 if they're within `nan_interp_max_dist` pixels (default 3). Good for
 isolated holes and one-pixel mosaic seams. Tiles whose holes are too
 big or too far from real pixels are still dropped.
- **`mask`** — keep the tile, replace NaNs with 0 in the data bands, and
 append a binary `valid_mask` channel so your loss can ignore them
 ("pad-and-ignore"). The standard approach for U-Net-style segmentation.

Below we tile the same Sentinel-2 scene three times with the same
`cloud_mask=True` setting and compare the resulting tile counts.

In [ ]:
S2_TILES_DIR = TILES / "s2"
if S2_TILES_DIR.exists():
    shutil.rmtree(S2_TILES_DIR)

# 64-px tiles strike a balance between speed and showing real tile structure
# on the configured AOI. Resize via TILE_SIZE if you have changed the AOI in cell 6.
TILE_SIZE  = 64
STRIDE     = 64

results = {}
for mode in ("drop", "interpolate", "mask"):
    out = S2_TILES_DIR / mode
    print(f"\n--- nan_handling = {mode!r} ---")
    tile_geotiff(
        input_tiff=tiff_s2,
        output_dir=str(out),
        tile_size=TILE_SIZE, stride=STRIDE,
        augment=False, output_mode="geotiff",
        train_val_test_split=(0.8, 0.1, 0.1),
        split_method="random",
        nan_handling=mode,
        cloud_mask=True,
    )
    results[mode] = count_tiles(out)

print("\nTile counts per nan_handling mode (cloud_mask=True):")
print(f"{'mode':>12s}  {'train':>6s}  {'val':>6s}  {'test':>6s}  {'total':>6s}")
for mode, c in results.items():
    t = sum(c.values())
    print(f"{mode:>12s}  {c['train']:>6d}  {c['val']:>6d}  {c['test']:>6d}  {t:>6d}")


In [ ]:
# --- visualise: for ONE meaningful tile per mode, show before/after NaN handling ---
# The tiler writes a tiles_metadata.csv with per-tile bookkeeping
# (n_nan_before, n_filled, has_mask_band, n_cloud_masked). best_demo_tile()
# uses that file to pick the tile that ACTUALLY shows the mode in action:
#   - drop:        the kept tile with the highest n_nan_before
#                  (i.e. the cloudiest one that barely survived the threshold)
#   - interpolate: the tile with the highest n_filled
#   - mask:        the tile with the most cloud-masked pixels AND a valid_mask band
# Without this, a randomly picked tile is almost always cloud-free and the
# figure is uninformative.

fig, axes = plt.subplots(3, 3, figsize=(11.5, 9.5), constrained_layout=True)
fig.suptitle("NaN handling -- B04 (stretched), NaN map, validity mask (if any)",
             fontsize=12)

for row, mode in enumerate(("drop", "interpolate", "mask")):
    tp, n_nan_before = best_demo_tile(S2_TILES_DIR / mode, mode)
    if tp is None:
        for c in range(3): axes[row][c].axis("off")
        axes[row][0].set_title(f"{mode!r}: no tiles kept")
        continue
    arr, descs, tags = open_tile(tp)
    bi_b04 = descs.index("B04")
    b04 = arr[bi_b04]
    nan_map = np.isnan(b04)
    tile_n_px = b04.size
    pct_before = 100.0 * n_nan_before / tile_n_px
    pct_after  = 100.0 * nan_map.mean()
    axes[row][0].imshow(stretch01(b04), cmap="gray")
    axes[row][0].set_title(f"{mode!r}  -  B04 (stretched)\n"
                           f"NaN: {pct_before:.1f}% before, {pct_after:.1f}% after  "
                           f"({tp.name})", fontsize=9)
    axes[row][1].imshow(nan_map, cmap="Greys")
    axes[row][1].set_title("NaN map (white = NaN)", fontsize=9)
    if "valid_mask" in descs:
        bi_vm = descs.index("valid_mask")
        axes[row][2].imshow(arr[bi_vm], cmap="Greys_r")
        axes[row][2].set_title("appended valid_mask band", fontsize=9)
    else:
        axes[row][2].imshow(np.zeros_like(b04), cmap="Greys")
        axes[row][2].set_title("(no valid_mask band)", fontsize=9)
    for ax in axes[row]:
        ax.set_xticks([]); ax.set_yticks([])

plt.show()


Read the three rows top to bottom:

- **`drop`** — every kept tile is completely NaN-free, by construction.
- **`interpolate`** — the NaN map should be near-empty for the kept tiles
 (any small holes were filled from neighbours). The pipeline reports how
 many pixels it filled per tile in the embedded `tile_n_filled` tag.
- **`mask`** — NaN pixels are replaced by zeros in the data bands and the
 appended `valid_mask` band remembers where they were. Use that band as
 a loss mask in PyTorch / TensorFlow.

Rule of thumb: start with `interpolate` for small ROIs and `mask` for
segmentation models; reserve `drop` for very clean training sets where
you have plenty of area.

## 7. Tiling — with and without overlap

Models train on small square patches, not on whole scenes. `tile_geotiff`
slides a window of `tile_size` pixels across the GeoTIFF in steps of
`stride` pixels:

- **`stride = tile_size`** — disjoint tiles. Simplest, fastest, smallest
 dataset. Adjacent tiles share zero pixels.
- **`stride = tile_size // 2`** — 50% overlap. ~4× as many tiles, each
 pixel appears in up to 4 tiles. Useful when you want to augment the
 effective training set or to do test-time tiling with averaging in the
 overlaps.

Below we tile the Sentinel-2 scene both ways and overlay the tile grid on
the source image, then show a 4×4 sample of the resulting tiles.

In [ ]:
# Tile twice: no overlap and 50% overlap, no cloud masking, no augmentation.
for tag, stride in (("no_overlap", TILE_SIZE), ("overlap50", TILE_SIZE // 2)):
    out = TILES / f"s2_{tag}"
    if out.exists(): shutil.rmtree(out)
    tile_geotiff(
        input_tiff=tiff_s2, output_dir=str(out),
        tile_size=TILE_SIZE, stride=stride,
        augment=False, output_mode="geotiff",
        train_val_test_split=(1.0, 0.0, 0.0),
        split_method="random", nan_handling="mask",
    )

# Count tiles
n_no  = len(list((TILES / "s2_no_overlap").glob("train/*.tif")))
n_ov  = len(list((TILES / "s2_overlap50").glob("train/*.tif")))
print(f"\nTile counts:  no_overlap = {n_no}   |   50% overlap = {n_ov}"
      f"(ratio = {n_ov / max(n_no,1):.1f}x)")

In [ ]:
# --- overlay the tile grid on the source image, two panels: no overlap vs 75% overlap ---
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
tile_grid_overlay(arr_s2, descs_s2, TILE_SIZE, TILE_SIZE, axes[0],
                  f"stride = tile_size = {TILE_SIZE}  (no overlap)")
# 75% overlap (stride = tile_size/4) is dramatic enough that the
# overlap structure is unmistakable when paired with the 4-colour palette.
tile_grid_overlay(arr_s2, descs_s2, TILE_SIZE, TILE_SIZE // 4, axes[1],
                  f"stride = {TILE_SIZE//4}  (75% overlap)")
plt.tight_layout(); plt.show()


In [ ]:
# --- show a 4x4 grid of sample tiles from the no-overlap run ---
tile_paths = sorted((TILES / "s2_no_overlap" / "train").glob("*.tif"))
n_show = min(16, len(tile_paths))
if n_show == 0:
    print("(no tiles to show — AOI may be smaller than tile_size)")
else:
    cols = 4
    rows = int(np.ceil(n_show / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.6 * rows))
    for i in range(rows * cols):
        ax = axes.flat[i] if rows > 1 else axes[i]
        if i >= n_show:
            ax.axis("off"); continue
        with rasterio.open(tile_paths[i]) as src:
            arr = src.read()
            descs = list(src.descriptions or [])
        bi = descs.index("B04") if "B04" in descs else 0
        ax.imshow(stretch01(arr[bi]), cmap="gray")
        ax.set_title(os.path.basename(tile_paths[i]), fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Sample tiles (no overlap, B04 stretched)", y=1.00)
    plt.tight_layout(); plt.show()

## 8. Train / val / test split strategies

For satellite imagery, **how** you split tiles matters as much as *what
ratio* you use. Neighbouring tiles often share the same clouds, the same
crop fields, the same shadows — so a random split causes **spatial
data leakage** that makes your test metrics rosier than they should be.

The tiler offers four strategies, all of them reproducible across runs:

- **`random`** — coin flip per tile. Maximum leakage. Use only as a
 sanity-check baseline.
- **`block`** (the **default**) — partition the scene into
 `block_size_tiles × block_size_tiles` blocks (e.g. 4×4 = ~10 km on a
 10 m grid) and assign each whole block to one split. The standard
 spatial-CV approach.
- **`stripes`** — like `block` but 1-D: contiguous N-row (or N-column)
 bands go to one split. Useful when your gradient runs along an axis
 (e.g. elevation, urbanisation).
- **`regions`** — explicit per-split AOIs in the same `aoi.py` spec
 language as `main.py`. The cleanest separation, the most manual setup.

For the **why** behind spatial CV in EO, see
[Hsu & Moortgat 2026 (Remote Sens. 18:1768)](https://www.mdpi.com/search?q=Hsu+Moortgat+bathymetry).

In [ ]:
# We need a few more tiles to see the split patterns clearly, so use a denser
# stride for this exercise.
SPLITS_DIR = TILES / "splits"
if SPLITS_DIR.exists(): shutil.rmtree(SPLITS_DIR)

common = dict(
    input_tiff=tiff_s2,
    tile_size=TILE_SIZE, stride=TILE_SIZE // 2,
    augment=False, output_mode="geotiff",
    train_val_test_split=(0.7, 0.15, 0.15),
    nan_handling="mask",
)

print("--- random ---")
tile_geotiff(output_dir=str(SPLITS_DIR / "random"), split_method="random", **common)

print("\n--- block (2x2 tile blocks) ---")
tile_geotiff(output_dir=str(SPLITS_DIR / "block"), split_method="block",
             split_block_size_tiles=2, **common)

print("\n--- stripes (horizontal, 1-tile rows) ---")
tile_geotiff(output_dir=str(SPLITS_DIR / "stripes"), split_method="stripes",
             split_stripe_axis="horizontal", split_stripe_size_tiles=1, **common)

# regions: the cleanest geographic separation -- send each city to its
# own split bucket. Columbus stays as training data; Cincinnati becomes
# the validation set; Cleveland becomes the test set. This is the kind
# of cross-city setup you would use to ask "does this model that learned
# from one city generalise to another?"
print("\n--- regions (Columbus=train, Cincinnati=val, Cleveland=test) ---")
AOI_CIN = {"center": (39.103, -84.512), "side_miles": 4}
AOI_CLE = {"center": (41.500, -81.694), "side_miles": 4}
ROI_CIN = resolve_aoi(AOI_CIN)
ROI_CLE = resolve_aoi(AOI_CLE)

# Fetch S2 for the two extra cities (Columbus is already fetched as tiff_s2)
print(">>> Cincinnati <<<")
fetch_sentinel_data(
    "Sentinel-2", BANDS_S2, TIME_RANGE_OPTICAL, ROI_CIN,
    resolution=RES_S2, save_folder=str(DATA),
    max_cloud_coverage=MAX_CLOUD, min_cloud_coverage=MIN_CLOUD, provider="auto",
)
CIN_DIR = sorted(DATA.glob("Sentinel-2_*"),
                 key=lambda p: (os.path.getmtime(p), str(p)))[-1]
tiff_s2_cin = sorted(glob.glob(str(CIN_DIR / "*_full_size.tiff")))[0]

print(">>> Cleveland <<<")
fetch_sentinel_data(
    "Sentinel-2", BANDS_S2, TIME_RANGE_OPTICAL, ROI_CLE,
    resolution=RES_S2, save_folder=str(DATA),
    max_cloud_coverage=MAX_CLOUD, min_cloud_coverage=MIN_CLOUD, provider="auto",
)
CLE_DIR = sorted(DATA.glob("Sentinel-2_*"),
                 key=lambda p: (os.path.getmtime(p), str(p)))[-1]
tiff_s2_cle = sorted(glob.glob(str(CLE_DIR / "*_full_size.tiff")))[0]

# Tile each city, force all of its tiles into the chosen bucket, and
# move them into a single SPLITS_DIR/regions tree. We use (1, 0, 0) /
# (0, 1, 0) / (0, 0, 1) so the random draw inside the tiler always
# resolves to the same bucket -- effectively "this scene's tiles all
# belong to the target split".
REGIONS_DIR = SPLITS_DIR / "regions"
for sub in ("train", "val", "test"):
    (REGIONS_DIR / sub).mkdir(parents=True, exist_ok=True)
city_inputs = [
    ("train", tiff_s2),     # Columbus
    ("val",   tiff_s2_cin), # Cincinnati
    ("test",  tiff_s2_cle), # Cleveland
]
ratio_for_bucket = {"train": (1.0, 0.0, 0.0),
                    "val":   (0.0, 1.0, 0.0),
                    "test":  (0.0, 0.0, 1.0)}
for bucket, city_tiff in city_inputs:
    tmp_out = REGIONS_DIR / f"_tmp_{bucket}"
    tile_geotiff(
        input_tiff=city_tiff, output_dir=str(tmp_out),
        tile_size=TILE_SIZE, stride=TILE_SIZE,
        augment=False, output_mode="geotiff",
        train_val_test_split=ratio_for_bucket[bucket],
        split_method="random",
        nan_handling="mask",
    )
    # All tiles landed in tmp_out/<bucket>/ thanks to the (1,0,0)-style ratio.
    src_dir = tmp_out / bucket
    for f in sorted(src_dir.glob("*.tif")):
        shutil.move(str(f), str(REGIONS_DIR / bucket / f.name))
    shutil.rmtree(tmp_out)

print("\nSplit counts per method:")
print(f"{'method':>10s}  {'train':>6s}  {'val':>6s}  {'test':>6s}")
for m in ("random", "block", "stripes", "regions"):
    c = count_tiles(SPLITS_DIR / m)
    print(f"{m:>10s}  {c['train']:>6d}  {c['val']:>6d}  {c['test']:>6d}")


In [ ]:
# --- plot the spatial layout of each split method ---
# Top row: random / block / stripes overlaid on the same Columbus image
# (head-to-head comparison). The regions case is fundamentally cross-city
# now, so it gets its own row -- one panel per city, each coloured by
# its split bucket.

# Top figure: random / block / stripes on the same Columbus image
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
plot_split_layout(arr_s2, descs_s2, "random",  SPLITS_DIR,
                  axes[0], "random  -- coin flip per tile")
plot_split_layout(arr_s2, descs_s2, "block",   SPLITS_DIR,
                  axes[1], "block   -- 2x2 tile blocks")
plot_split_layout(arr_s2, descs_s2, "stripes", SPLITS_DIR,
                  axes[2], "stripes -- 1-tile horizontal rows")
fig.suptitle("Within-AOI split strategies (over Columbus)", y=1.02)
plt.tight_layout(); plt.show()

# Bottom figure: regions case -- one city per bucket
arr_cin, descs_cin, *_ = open_response(CIN_DIR)
arr_cle, descs_cle, *_ = open_response(CLE_DIR)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
city_panel(arr_s2,  descs_s2,  "Columbus",   "train", axes[0])
city_panel(arr_cin, descs_cin, "Cincinnati", "val",   axes[1])
city_panel(arr_cle, descs_cle, "Cleveland",  "test",  axes[2])
fig.suptitle("regions -- each city is its own split (Columbus=train, Cincinnati=val, Cleveland=test)",
             y=1.02)
plt.tight_layout(); plt.show()


Notice how *random* speckles train/val/test pixels next to each other (=
leakage) while *block* and *stripes* keep each split in its own contiguous
patch of the Columbus scene. **regions** is the strongest separation of
all because train, val, and test are not even in the same city -- the
network never gets to "look over the fence" at the validation set during
training. Cross-city evaluation is the closest you can get to honestly
measuring "will my model generalise to a place it has never seen?".

> **Worth knowing.** If you swap the AOI in cell 6 for something *much*
> smaller than the 10 mi default, you may see the warning
> `⚠️ Empty bucket(s): ...` in some of the prints above. That is the
> tiler telling you the tile grid is too coarse to spatially separate
> all three buckets -- it will name the exact knob to turn
> (`split_block_size_tiles`, `split_method`, AOI size). Auto-scaling
> the block size makes this rare on real-world AOIs.


## 9. Multi-mission fusion

`fusion.fuse_response_tiffs` takes a list of per-mission `<Mission>_full_size.tiff`
files and welds them onto **one common UTM grid at one resolution**.
Bands keep their provenance via a `<Mission>_<BandName>` prefix
(e.g. `Sentinel-2_B04`, `Sentinel-1_VV`, `Landsat_BQA`). The default is
to use the **intersection** of input footprints — the safe choice for
per-pixel multi-modal models, since every output pixel has every input.

Below we fuse Sentinel-2, Sentinel-1 (if available), and the DEM onto a
10 m grid. We then print the fused cube's bands, shape, CRS, and pixel
size and show one band per mission side by side so you can visually
confirm they're on the same grid.

In [ ]:
# --- assemble whatever inputs we successfully fetched ---
fuse_inputs = [tiff_s2]
if S1_DIR is not None:
    fuse_inputs.append(find_response(S1_DIR))
if LSC_DIR is not None:
    fuse_inputs.append(find_response(LSC_DIR))
fuse_inputs.append(find_response(DEM_DIR))
fuse_inputs.append(find_response(LULC_DIR))

print("Fusing inputs:")
for p in fuse_inputs:
    with rasterio.open(p) as src:
        print(f"{os.path.relpath(p, OUT):60s}"
              f"bands={src.count}  shape={src.shape}  CRS={src.crs}")

fused_path = str(FUSE / "columbus_cube.tiff")
fused = fuse_response_tiffs(
    inputs=fuse_inputs,
    output_path=fused_path,
    resolution=10,        # 10 m common grid
    dst_crs=None,         # default = CRS of the first input (S2's UTM zone)
    bbox_mode="intersection",
)

print("\nFused cube summary:")
print(f"bands ({len(fused['bands'])}):")
for b in fused["bands"]:
    print(f"{b}")
print(f"shape (C,H,W) : {fused['shape']}")
print(f"CRS           : {fused['crs']}")
with rasterio.open(fused_path) as src:
    px_x, px_y = src.transform.a, -src.transform.e
    print(f"pixel size    : {px_x:g} x {px_y:g}  (CRS units, metres)")
    print(f"file size     : {os.path.getsize(fused_path) / 1024:.1f} KB")


In [ ]:
# --- visualise one band per mission to confirm they share the same grid ---
with rasterio.open(fused_path) as src:
    fused_arr   = src.read()
    fused_descs = list(src.descriptions or [])

picks = []
for label, suffixes, cm in [
    ("Sentinel-2 B04", ("B04",),       "gray"),
    ("Sentinel-1 VV",  ("VV",),        "gray"),
    ("Landsat B04",    ("B04",),       "gray"),     # may be None if Landsat skipped
    ("DEM",            ("DEM",),       "terrain"),
    ("WorldCover",     ("LULC",),      "tab20"),
]:
    # pick by mission-prefixed name
    target = None
    for i, d in enumerate(fused_descs):
        if d and d.endswith("_" + suffixes[0]):
            mission = d.rsplit("_", 1)[0]
            if label.startswith(mission):
                target = i; break
    if target is not None:
        picks.append((target, fused_descs[target], cm))

ncols = min(len(picks), 5)
nrows = int(np.ceil(len(picks) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows + 0.4))
if nrows == 1: axes = np.array([axes])
for ax in axes.flat:
    ax.axis("off")
for i, (bi, name, cm) in enumerate(picks):
    ax = axes.flat[i]
    ax.axis("on")
    band = fused_arr[bi]
    if cm in ("gray",):
        ax.imshow(stretch01(band), cmap=cm)
    else:
        ax.imshow(band, cmap=cm)
    ax.set_title(name, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("One band per mission, all on the same 10 m UTM grid", y=1.00)
plt.tight_layout(); plt.show()


Every panel above has the **same shape, same CRS, same pixel size**. The
fused TIFF is now ready to be tiled and fed to a multi-modal model, with
each band traceable back to its original mission via its prefix.

The sidecar JSON next to the fused TIFF records the bands list, the
output CRS, the resolution, and the source paths — handy for downstream
provenance tracking.

## 10. Reading metadata back from a tile

Every tile the pipeline writes carries two kinds of provenance information
**embedded as GeoTIFF tags** (no sidecar required):

- **`source_*`** tags — copied from the source scene's `userdata.json`
 (satellite, acquisition date, scene id, provider, collection) and, for
 fused tiles, from the fusion's `.meta.json` sidecar (band list, source
 paths, fusion bbox mode, resolution).
- **`tile_*`** tags — the parameters of *this particular* tile
 (`tile_window_x`, `tile_window_y`, `tile_split_method`,
 `tile_split_assigned`, `tile_nan_handling`, `tile_n_nan_before`,
 `tile_n_filled`, `tile_n_cloud_masked`, `tile_augmentation`, ...).

You read them back with `rasterio.open(tile).tags()`. The band names live
on `src.descriptions`.

In [ ]:
# pick the first tile of the 'block' split — walk train/val/test in order
# because the block assignment can land tiles in any single bucket -- walk all three.
tile_paths = []
for bucket in ("train", "val", "test"):
    tile_paths = sorted((SPLITS_DIR / "block" / bucket).glob("*.tif"))
    if tile_paths:
        break
assert tile_paths, "No tiles produced by the block-split run — re-run section 7 first."
sample_tile = tile_paths[0]
print(f"Inspecting:  {os.path.relpath(sample_tile, OUT)}\n")

with rasterio.open(sample_tile) as src:
    tags  = src.tags()
    descs = list(src.descriptions or [])
    print(f"shape (C,H,W) : ({src.count}, {src.height}, {src.width})")
    print(f"CRS           : {src.crs}")
    print(f"transform     : {src.transform}")
    print(f"band descriptions:")
    for i, d in enumerate(descs, 1):
        print(f"band {i}: {d}")

# pretty-print source_* vs tile_* keys
sources = {k: v for k, v in tags.items() if k.startswith("source_")}
tiles   = {k: v for k, v in tags.items() if k.startswith("tile_")}
others  = {k: v for k, v in tags.items() if not (k.startswith("source_") or k.startswith("tile_"))}

print("\nsource_* tags  (where did this tile come from?):")
for k, v in sources.items():
    print(f"{k:>28s} = {v}")

print("\ntile_* tags  (how was this tile cut?):")
for k, v in tiles.items():
    print(f"{k:>28s} = {v}")

if others:
    print("\nother tags:")
    for k, v in others.items():
        print(f"{k:>28s} = {v}")

Useful follow-ups:

- **Group tiles by source scene** — just sort by `source_tileId`.
- **Filter out augmented variants** — `tile_augmentation == "none"`.
- **Reconstruct the original split** — read `tile_split_method` and
 `tile_split_assigned`. No need to keep a separate manifest file.
- **Audit data quality** — `tile_n_nan_before`, `tile_n_filled`,
 `tile_n_cloud_masked` tell you how many pixels were touched.

## 11. Augmentation

Pass `augment=True` to `tile_geotiff` and every tile gains **five
augmented variants**: horizontal flip, vertical flip, 90° rotation,
270° rotation, and additive Gaussian noise. Each variant carries the
same `source_*` tags as the parent (so provenance is preserved) plus its
own `tile_augmentation` tag (`flipH`, `flipV`, `rot90`, `rot270`, `noise`).

In [ ]:
AUG_DIR = TILES / "s2_augmented"
if AUG_DIR.exists(): shutil.rmtree(AUG_DIR)

tile_geotiff(
    input_tiff=tiff_s2,
    output_dir=str(AUG_DIR),
    tile_size=TILE_SIZE, stride=TILE_SIZE,
    augment=True, output_mode="geotiff",
    train_val_test_split=(1.0, 0.0, 0.0),
    split_method="random",
    nan_handling="mask",
)

# pick the parent tile of the first set of augmented variants
parents = sorted((AUG_DIR / "train").glob("tile_*.tif"))
parents = [p for p in parents if "_" not in p.stem[len("tile_00000"):]]  # exclude augmented
parent = parents[0]
# the augmented siblings share the parent's tile_id prefix
sibs = sorted((AUG_DIR / "train").glob(f"{parent.stem}_*.tif"))
print(f"Parent tile : {parent.name}")
print(f"Augmented variants ({len(sibs)}):")
for s in sibs:
    print(f"{s.name}")

In [ ]:
# --- show parent + each augmented variant ---
all_tiles = [parent] + sibs
fig, axes = plt.subplots(1, len(all_tiles), figsize=(2.6 * len(all_tiles), 3.0))
for ax, p in zip(axes, all_tiles):
    with rasterio.open(p) as src:
        arr   = src.read()
        descs = list(src.descriptions or [])
        tags  = src.tags()
    bi = descs.index("B04") if "B04" in descs else 0
    ax.imshow(stretch01(arr[bi]), cmap="gray")
    label = tags.get("tile_augmentation", "?")
    ax.set_title(label, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("One tile and its five augmented siblings (B04 band)", y=1.05)
plt.tight_layout(); plt.show()

# confirm provenance is preserved
with rasterio.open(sibs[0]) as src:
    src_tags = src.tags()
with rasterio.open(parent) as src:
    par_tags = src.tags()
shared = {k: v for k, v in src_tags.items() if k.startswith("source_")}
print(f"\nShared source_* keys between augmented variant and parent: {len(shared)}")
print(f"e.g. source_satellite = {shared.get('source_satellite')}")
print(f"\nparent.tile_augmentation     = {par_tags.get('tile_augmentation')}")
print(f"augmented.tile_augmentation  = {src_tags.get('tile_augmentation')}")

## 12. Exporting for training (Zarr / LMDB)

For real training runs you usually don't want thousands of tiny GeoTIFFs
— I/O overhead dominates. The pipeline ships two export helpers that
roll the tiles into chunked, GPU-friendly stores:

- **`export_zarr.py`** — writes a `tiles.zarr` group with one chunked
 array per split plus a `metadata.json`. Plays nicely with `dask`,
 `xarray`, and the `dataset_loader.py` PyTorch dataset.
- **`export_lmdb.py`** — writes a single LMDB key-value store. Tiny
 random-read latency, ideal for shuffled epoch iteration on a single
 GPU box.

We don't run them here (a real export to Zarr can take a minute or two
and we want this notebook fast); see those scripts for the exact CLI.
The Zarr cube that ships with the repo for the
[`01_water_classification.ipynb`](01_water_classification.ipynb) notebook was
built with `export_zarr.py`.

## 13. Doing the same thing from plain Python (`python main.py`)

Everything you've seen here can be reproduced from a single command-line
invocation. Open `geoai_datacubes/main.py`, edit the
`USER INPUT` block at the top to look like:

```python
PROVIDER = "auto"
MISSION = "Sentinel-2"
BANDS = None
AOI = {"center": (40.0067, -83.0305), "side_miles": 10} # OSU, 10 mi square
ROI = resolve_aoi(AOI)
TIME_RANGE = ("2024-06-01", "2024-06-30")
RESOLUTION = 10
MAX_CLOUD = 0.20
TILE_SIZE = 64
SPLIT = (0.8, 0.1, 0.1)
```

Then:

```bash
python -m geoai_datacubes.main
```

`main.py` does the same thing as this notebook's sections 3 (fetch) + 5
(cloud mask) + 6 (NaN handling) + 7-8 (tile + split): it writes
`data/<Mission>_<date>_<scene_id>/<Mission>_full_size.tiff`, computes NDVI, masks
clouds, tiles into `data/<scene>/tiles_v2/{train,val,test}/`, and saves a
quick-look PNG.

To run several missions in one shot, drive the pipeline from a small bash
loop:

```bash
for M in Sentinel-2 Sentinel-1 Landsat Copernicus-DEM ESA-WorldCover; do
 sed -i.bak "s/^MISSION.*/MISSION = '$M'/" main.py
 python main.py
done
```

The notebook is great for exploration; plain Python is what you want for
production, CI, and batch scheduling.

## 14. Running on an HPC cluster (SLURM)

SLURM is the job scheduler used by most academic HPC clusters. You write
a short shell script that begins with `#SBATCH` directives describing
the resources you need (CPUs, memory, walltime, partition) and ends with
your actual command (`python main.py`). The cluster's scheduler queues
your job, runs it on a compute node when resources are free, and writes
stdout/stderr to log files. Three commands cover 90% of the workflow:
`sbatch script.sbatch` (submit), `squeue -u $USER` (check status),
`scancel <jobid>` (cancel).

Two **generic** templates live in [`slurm_examples/`](../slurm_examples/):

- [`single_fetch.sbatch`](../slurm_examples/single_fetch.sbatch) — one
 job, one mission, one AOI; the cluster equivalent of running
 `python main.py` locally.
- [`array_fetch.sbatch`](../slurm_examples/array_fetch.sbatch) — a SLURM
 **job array** that fetches many (mission, time-range) tuples in
 parallel from a single submission. The tuple table lives in the script
 itself; edit it, update `--array=0-N`, submit once.

Both have placeholder `--account`, `--partition`, and `--mail-user`
values — fill in your cluster's values before submitting. The templates
deliberately do not name any cluster (Unity, OSC, Pitzer, etc.) so they
work anywhere SLURM does.

For cluster-specific guidance (module names, partition lists, JupyterHub
URLs, conda activation patterns, GPU partitions, fairshare) see the
[**BuckAI HPC Handbook**](https://github.com/buckai-observatory/buckai-hpc-handbook).

## 15. Where to go next

You now have:
- A working data cube over Columbus (every mission you fetched, fused
 onto a common UTM grid).
- Tiled, split, optionally cloud-masked and NaN-handled training data.
- Embedded provenance in every tile.

Next steps:

- **Train a model** on the cube. The companion notebook
 [`01_water_classification.ipynb`](01_water_classification.ipynb) walks through
 a tiny PyTorch U-Net on a bundled Zarr cube; swap in your own cube to
 get started.
- **Scale up.** Pick a bigger AOI (the `center+side_miles` format makes
 this easy), a longer time range, and run on an HPC node using the
 SLURM templates.
- **Add a new mission.** Drop a new entry in
 `geoai_datacubes/fetch/missions.py` describing the STAC collection,
 asset map, default bands, and (optionally) cloud-mask rule. The rest
 of the pipeline picks it up automatically.
- **Report bugs / feature requests** on
 [github.com/buckai-observatory/geoai-datacubes](https://github.com/buckai-observatory/geoai-datacubes).
- **Get in touch** with the BuckAI Observatory at
 [buckai-observatory.org](https://buckai-observatory.org) — we're
 happy to help with new applications, point you at related tools, or
 bring you in as a collaborator.

Happy modelling.